#Training a MiniGpt Model

In [30]:
"""
MIni GPT trainin script (PyTorch).
Requirements:
    pip install torch transformers datasets tqdm.
Usage examples:
    python mini_gpt_train.py --dataset wikiext --dataset_config wikitext-2-raw-v1
    python mini_gpt_train.py --text_file my_corpus.txt
"""
import argparse
import math
import os
from pathlib import Path
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import GPT2TokenizerFast
from datasets import load_dataset
from tqdm.auto import tqdm




In [31]:
# Small  Decoder only Transformer
class CasualSelfAttention(nn.Module):
  def __init__(self, n_embd, n_head, attn_pdrop=0.0, resid_pdrop=0.0):
    super().__init__()
    assert n_embd % n_head == 0
    self.n_head = n_head
    self.head_dim = n_embd // n_head
    self.scale = self.head_dim ** - 0.5

    self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
    self.out =  nn.Linear(n_embd, n_embd, bias=False)
    self.attn_drop = nn.Dropout(attn_pdrop)
    self.resid_drop  = nn.Dropout(resid_pdrop)

  def forward(self, x , mask=None):
    B, T, C = x.size()
    qkv = self.qkv(x)   # (B, T, 3*C)
    q, k, v = qkv.split(c, dim=2)
    # reshape for multi-head (B, n_head, T, head_dim)
    q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
    k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
    v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

    # scaled dot product
    att = (q @ k.transpose(-2, -1)) * self.scale    # (B, nh , T, T)
    # casual mask: allow attending to past only
    casual_mask = torch.tril(torch.ones(T, T, device=x.device)).unsqueeze(0)
    att = att.masked_fill(casual_mask == 0, float("-inf"))

    if mask is not None:
      # mask shape expected ( B, 1, 1, T ) or boradastable
      att = att.masked_fill(mask ==0, float("-inf"))

    att = F.softmax(att, dim=1)
    att = self.attn_drop(att)
    y = att @ v # (B, nh, T, head_dim)
    y = y.transpose(1, 2).contiguous().view(B, T, C)
    y =self.resid_drop(self.out(y))
    return y






In [32]:
class FeedForward(nn.Module):
  def __init__(self, n_embd, n_ff, resid_pdrop=0.0):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embd, n_ff),
        nn.GELU(),
        nn.Linear(n_ff, n_embd),
        nn.Dropout(resid_pdrop),
    )

  def forward(self, x):
    return self.net(x)




In [33]:
class Block(nn.Module):
  def __init__(self, n_embd, n_head, n_ff, attn_pdrop, resid_pdrop):
    super().__init__()
    self.ln1 = nn.LayerNorm(n_embd, eps=1e-5)
    self.attn = CasualSelfAttention(n_embd, n_head, attn_pdrop, resid_pdrop)
    self.ln2 = nn.LayerNorm(n_embd, eps=1e-5)
    self.ff = FeedForward(n_embd, n_ff, resid_pdrop)

  def forward(self, x, mask=None):
    x = x + self.attn(self.ln1(x), mask=mask)
    x = x + self.ff(self.ln2(x))
    return x



In [34]:
class MiniGPT(nn.Module):
  def __init__(self, vocab_size, seq_len, n_layer=6, n_head=8, n_embd=512, n_ff=None, attn_pdrop=0.0, resid_pdrop=0.0):
    super().__init__()
    n_eff = n_eff or 4 * n_embd
    self.vocab_size = vocab_size
    self.seq_len = seq_len
    self.tok_emb =  nn.Embedding(vocab_size, n_embd)
    self.ops_emb = nn.Parameter(torch.zeros(1, seq_len, n_embd))
    self.drop = nn.Dropout(0.0)
    self.blocks = nn.Sequential(*[Block(n_embd, n_head, n_ff, attn_pdrop,resid_pdrop) for _ in range(n_layer)])
    self.ln_f = nn.LayerNorm(n_embd, eps=1e-5)
    # weight tying : lm_head weight = tok_emb weight transpose
    self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

    # Iinitalization
    self.apply(self._init_weights)

  def _init_weights(self, module):
    if isinstance(module, (nn.Linear, nn.Embedding)):
      nn.init.normal_(module.weight, mean=0.0, stf=0.02)
    if isinstance(module, nn.Linear) and module.bias is not None:
      nn.init.zeros_(module.bias)

  def forward(self, idx):
    """ indx: (B, T) int tokens
        returns logits (B, T, V)
    """
    B, T = idx.size()
    assert  T <= self.seq_len, "Sequence length exceeds model capacity"
    tok = self.tok_emb(idx)   # ( B, T, C)
    pos = self.pos_emb[: T, :]   # (1, T, C)
    x = self.drop(tok + pos)
    # Casual mask is handled indife the attention with triangular mask , so no mask parameter is required
    x = self.blocks(x)
    x = self.ln_f(x)
    logits = self.lm_head(x)
    return logits

  def generate(self, idx, max_new_tokens, eos_token_id: Optional[int] = None, temperature=1.0):
    """ Autoregressive generae (naive)
        idx: (B, T)
    """
    self.eval()
    device = next(self.parameters()).device
    idx = idx.too(device)
    B, T = idx.size()
    for _ in range(max_new_tokens):
      Tcur = idx.size(1)
      if Tcur > self.seq_len:
        idx = idx[:, -self.seq_len :]
        Tcur = self.seq_len

      with torch.no_grad():
        logits = self(idx)
        logits = logits[:, -1, :] / max(1e-8, temperature)
        probs =  F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_token], dim=1)
        if eos_token_id is not None and (next_token == eos_token_id).all():
          break
    return idx






**Dataset helpers**

In [49]:
class TokenizedTextDataset(Dataset):
  def __init__(self, token_ids, seq_len):
    # token_ids: List or id torch tensor of token ids concatenated
    if isinstance(token_ids, list):
      token_ids = torch.tensor(token_ids, dtype=torch.long)
    self.data = token_ids
    self.seq_len = seq_len

  def  __len__(self):
    # Number of training examples (sliding windows)
    n = max(0, (self.data.size(0) -1 )// self.seq_len)
    return n

  def __getitem__(self, idx):
    start = idx * self.seq_len
    x = self.data[start: start + self.seq_len]
    y = self.data[start + 1 : start + 1 + self.seq_len]
    # If end-of-data pad with zeros (shouldn't usually happen for concatenated corpora)
    if x.size() < self.seq_len:
      pad_len = self.seq_len - x.size(0)
      x = F.pad(x, (0, pad_len), value=0)
      y = F.pad(y, (0, pad_len), value=0)
    return x, y

  def collate_batch(batch):
    xs = torch.stack([b[0] for b in batch], dim=0)
    ys = torch.stack([b[1] for b in batch], dim=0)
    return xs, ys






#Training Loop


In [52]:
def train(args):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    # ===== Tokenizer =====
    # We'll use GPT-2 tokenizer (byte-level BPE). It has a vocab size ~50k.
    tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
    tokenizer.add_special_tokens({"pad_token": "<|pad|>"})
    vocab_size = len(tokenizer)

    # ===== Dataset loading =====
    if args.text_file:
        text = Path(args.text_file).read_text(encoding="utf-8")
        data = {"train": [{"text": text}]}
        dataset = load_dataset("text", data_files={"train": args.text_file})
    else:
        # Example: wikitext-2-raw-v1 or other HF dataset
        ds = load_dataset(args.dataset, args.dataset_config) if args.dataset_config else load_dataset(args.dataset)
        # prefer 'train' split
        dataset = ds["train"] if "train" in ds else ds

    # Combine texts into one long string (simple approach)
    if args.text_file:
        texts = dataset["text"]
    else:
        # many datasets have a 'text' column or 'content'
        if isinstance(dataset, list):
            texts = [d.get("text", d.get("content", "")) for d in dataset]
        else:
            col = "text" if "text" in dataset.column_names else dataset.column_names[0]
            texts = dataset[col]

    # Optionally limit data for quick experiments
    if args.max_chars:
        concat_text = "".join(texts)[: args.max_chars]
    else:
        concat_text = "\n\n".join(texts)

    # Tokenize the entire corpus (fast)
    enc = tokenizer(concat_text, return_tensors="pt", add_special_tokens=False)["input_ids"].squeeze(0)
    print("Total tokens in corpus:", enc.size(0))

    dataset = TokenizedTextDataset(enc, args.seq_len)
    dataloader = DataLoader(dataset, batch_size=args.batch_size, shuffle=True, collate_fn=collate_batch, drop_last=True, num_workers=0)

    # ===== Model =====
    model = MiniGPT(
        vocab_size=vocab_size,
        seq_len=args.seq_len,
        n_layer=args.n_layer,
        n_head=args.n_head,
        n_embd=args.n_embd,
        n_ff=args.n_ff,
        attn_pdrop=args.attn_pdrop,
        resid_pdrop=args.resid_pdrop,
    )
    # tie weights
    model.lm_head.weight = model.tok_emb.weight
    model.to(device)
    print("Model params:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

        # ===== Training =====
    model.train()
    for epoch in range(args.epochs):

        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{args.epochs}", leave=False)
        total_loss = 0.0

        for step, (x, y) in enumerate(pbar):
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad(set_to_none=True)

            logits = model(x)  # (B, T, V)
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                y.view(-1)
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
            optimizer.step()

            if scheduler:
                scheduler.step()

            total_loss += loss.item()
            if (step + 1) % args.log_interval == 0:
                avg = total_loss / args.log_interval
                pbar.set_postfix_str(f"loss={avg:.4f}")
                total_loss = 0.0

        # ---- Save checkpoint per epoch ----
        out_dir = Path(args.output_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        ckpt_path = out_dir / f"mini_gpt_epoch{epoch+1}.pt"
        torch.save({
            "model_state_dict": model.state_dict(),
            "tokenizer_vocab": tokenizer.get_vocab()
        }, ckpt_path)
        print(f"Saved checkpoint: {ckpt_path}")

    # ===== Optional sample generation =====
    model.eval()
    prompt = args.sample_prompt or "Once upon a time"
    tokens = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)["input_ids"].to(device)

    with torch.no_grad():
        gen = model.generate(
            tokens,
            max_new_tokens=args.sample_length,
            eos_token_id=None,
            temperature=args.sample_temperature
        )

    text_out = tokenizer.decode(gen[0].cpu().numpy(), skip_special_tokens=True)
    print("--- sample ---")
    print(text_out)
    print("--------------")

    model.train()
    print("Training complete.")



In [53]:
# -------------------------------------
# CLI
# -------------------------------------
def parse_args():
  p = argparse.ArgumentParser()
  p.add_argument("--dataset", type=str, default="wikitext", help="HF dataset name (eg: 'wikitext')")
  p.add_argument("--dataset_config", type=str, default="wikitext-2-raw-v1", help="HF dataset config (e.g. 'wikitext-2-raw-v1')")
  p.add_argument("--text_file", type=str, default=None, help="Path to a single text file to train on (overrides dataset)")
  p.add_argument("--max_chars", type=int, default=2000000, help="Limit characters for quick runs (default 2M). Set to 0 for no limit.")
  p.add_argument("--seq_len", type=int, default=128)
  p.add_argument("--batch_size", type=int, default=32)
  p.add_argument("--epochs", type=int, default=3)
  p.add_argument("--lr", type=float, default=5e-4)
  p.add_argument("--weight_decay", type=float, default=0.01)
  p.add_argument("--grad_clip", type=float, default=1.0)
  p.add_argument("--n_layer", type=int, default=6)
  p.add_argument("--n_head", type=int, default=8)
  p.add_argument("--n_embd", type=int, default=512)
  p.add_argument("--n_ff", type=int, default=None)
  p.add_argument("--attn_pdrop", type=float, default=0.0)
  p.add_argument("--resid_pdrop", type=float, default=0.0)
  p.add_argument("--output_dir", type=str, default="./checkpoints")
  p.add_argument("--log_interval", type=int, default=20)
  p.add_argument("--onecycle", action="store_true", help="Use OneCycleLR scheduler")
  # sampling options
  p.add_argument("--sample_prompt", type=str, default="Once upon a time")
  p.add_argument("--sample_length", type=int, default=64)
  p.add_argument("--sample_temperature", type=float, default=1.0)

  # ==== FIX FOR COLAB / JUPYTER ====
  args, _ = p.parse_known_args()
  return args


if __name__ == "__main__":
  args = parse_args()
  # small tweak: if max_chars = 0 then no limit
  if args.max_chars == 0:
    args.max_chars = None

  train(args)


Using device: cpu


Token indices sequence length is longer than the specified maximum sequence length for this model (440294 > 1024). Running this sequence through the model will result in indexing errors


Total tokens in corpus: 440294


NameError: name 'collate_batch' is not defined

# Note
The error is because the code is intended to run as a single file I made it in the Colab.

# Let run it in a sinle file and see if we can fix it....!

In [3]:
# mini_gpt_train.py
"""
Mini GPT training script (PyTorch).
Requirements:
  pip install torch transformers datasets tqdm

Usage examples:
  python mini_gpt_train.py --dataset wikitext --dataset_config wikitext-2-raw-v1
  python mini_gpt_train.py --text_file my_corpus.txt
"""

import argparse
import math
import os
from pathlib import Path
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from transformers import GPT2TokenizerFast
from datasets import load_dataset
from tqdm.auto import tqdm


# ---------------------------
# Model: small Decoder-only Transformer
# ---------------------------
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, attn_pdrop=0.0, resid_pdrop=0.0):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.out = nn.Linear(n_embd, n_embd, bias=False)
        self.attn_drop = nn.Dropout(attn_pdrop)
        self.resid_drop = nn.Dropout(resid_pdrop)

    def forward(self, x, mask=None):
        B, T, C = x.size()
        qkv = self.qkv(x)  # (B, T, 3*C)
        q, k, v = qkv.split(C, dim=2)
        # reshape for multi-head: (B, n_head, T, head_dim)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        # scaled dot-product
        att = (q @ k.transpose(-2, -1)) * self.scale  # (B, nh, T, T)
        # causal mask: allow attending to past only
        causal_mask = torch.tril(torch.ones(T, T, device=x.device)).unsqueeze(0).unsqueeze(0)
        att = att.masked_fill(causal_mask == 0, float("-inf"))

        if mask is not None:
            # mask shape expected (B, 1, 1, T) or broadcastable
            att = att.masked_fill(mask == 0, float("-inf"))

        att = F.softmax(att, dim=-1)
        att = self.attn_drop(att)
        y = att @ v  # (B, nh, T, head_dim)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_drop(self.out(y))
        return y


class FeedForward(nn.Module):
    def __init__(self, n_embd, n_ff, resid_pdrop=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, n_ff),
            nn.GELU(),
            nn.Linear(n_ff, n_embd),
            nn.Dropout(resid_pdrop),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, n_embd, n_head, n_ff, attn_pdrop, resid_pdrop):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd, eps=1e-5)
        self.attn = CausalSelfAttention(n_embd, n_head, attn_pdrop, resid_pdrop)
        self.ln2 = nn.LayerNorm(n_embd, eps=1e-5)
        self.ff = FeedForward(n_embd, n_ff, resid_pdrop)

    def forward(self, x, mask=None):
        x = x + self.attn(self.ln1(x), mask=mask)
        x = x + self.ff(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(self, vocab_size, seq_len, n_layer=6, n_head=8, n_embd=512, n_ff=None, attn_pdrop=0.0, resid_pdrop=0.0):
        super().__init__()
        n_ff = n_ff or 4 * n_embd
        self.vocab_size = vocab_size
        self.seq_len = seq_len
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Parameter(torch.zeros(1, seq_len, n_embd))
        self.drop = nn.Dropout(0.0)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, n_ff, attn_pdrop, resid_pdrop) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd, eps=1e-5)
        # weight tying: lm_head weight = tok_emb weight transposed
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        # initialization
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if isinstance(module, nn.Linear) and module.bias is not None:
            nn.init.zeros_(module.bias)

    def forward(self, idx):
        """
        idx: (B, T) int tokens
        returns logits (B, T, V)
        """
        B, T = idx.size()
        assert T <= self.seq_len, "Sequence length exceeds model capacity"
        tok = self.tok_emb(idx)  # (B, T, C)
        pos = self.pos_emb[:, :T, :]  # (1, T, C)
        x = self.drop(tok + pos)
        # causal mask is handled inside attention with triangular mask, so no mask param required
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        return logits

    def generate(self, idx, max_new_tokens, eos_token_id: Optional[int] = None, temperature=1.0):
        """
        Autoregressive generate (naive).
        idx: (B, T)
        """
        self.eval()
        device = next(self.parameters()).device
        idx = idx.to(device)
        B, T = idx.size()
        for _ in range(max_new_tokens):
            Tcur = idx.size(1)
            if Tcur > self.seq_len:
                idx = idx[:, -self.seq_len :]
                Tcur = self.seq_len
            with torch.no_grad():
                logits = self(idx)  # (B, Tcur, V)
                logits = logits[:, -1, :] / max(1e-8, temperature)
                probs = F.softmax(logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)  # (B,1)
                idx = torch.cat([idx, next_token], dim=1)
                if eos_token_id is not None and (next_token == eos_token_id).all():
                    break
        return idx


# ---------------------------
# Dataset helpers
# ---------------------------
class TokenizedTextDataset(Dataset):
    def __init__(self, token_ids, seq_len):
        # token_ids: list or 1d torch tensor of token ids concatenated
        if isinstance(token_ids, list):
            token_ids = torch.tensor(token_ids, dtype=torch.long)
        self.data = token_ids
        self.seq_len = seq_len

    def __len__(self):
        # number of training examples (sliding windows)
        n = max(0, (self.data.size(0) - 1) // self.seq_len)
        return n

    def __getitem__(self, idx):
        start = idx * self.seq_len
        x = self.data[start : start + self.seq_len]
        y = self.data[start + 1 : start + 1 + self.seq_len]
        # If end-of-data pad with zeros (shouldn't usually happen for concatenated corpora)
        if x.size(0) < self.seq_len:
            pad_len = self.seq_len - x.size(0)
            x = F.pad(x, (0, pad_len), value=0)
            y = F.pad(y, (0, pad_len), value=0)
        return x, y


def collate_batch(batch):
    xs = torch.stack([b[0] for b in batch], dim=0)
    ys = torch.stack([b[1] for b in batch], dim=0)
    return xs, ys


# ---------------------------
# Training loop
# ---------------------------
def train(args):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    # ===== Tokenizer =====
    # We'll use GPT-2 tokenizer (byte-level BPE). It has a vocab size ~50k.
    tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
    tokenizer.add_special_tokens({"pad_token": "<|pad|>"})
    vocab_size = len(tokenizer)

    # ===== Dataset loading =====
    if args.text_file:
        text = Path(args.text_file).read_text(encoding="utf-8")
        data = {"train": [{"text": text}]}
        dataset = load_dataset("text", data_files={"train": args.text_file})
    else:
        # Example: wikitext-2-raw-v1 or other HF dataset
        ds = load_dataset(args.dataset, args.dataset_config) if args.dataset_config else load_dataset(args.dataset)
        # prefer 'train' split
        dataset = ds["train"] if "train" in ds else ds

    # Combine texts into one long string (simple approach)
    if args.text_file:
        texts = dataset["text"]
    else:
        # many datasets have a 'text' column or 'content'
        if isinstance(dataset, list):
            texts = [d.get("text", d.get("content", "")) for d in dataset]
        else:
            col = "text" if "text" in dataset.column_names else dataset.column_names[0]
            texts = dataset[col]

    # Optionally limit data for quick experiments
    if args.max_chars:
        concat_text = "".join(texts)[: args.max_chars]
    else:
        concat_text = "\n\n".join(texts)

    # Tokenize the entire corpus (fast)
    enc = tokenizer(concat_text, return_tensors="pt", add_special_tokens=False)["input_ids"].squeeze(0)
    print("Total tokens in corpus:", enc.size(0))

    dataset = TokenizedTextDataset(enc, args.seq_len)
    dataloader = DataLoader(dataset, batch_size=args.batch_size, shuffle=True, collate_fn=collate_batch, drop_last=True, num_workers=0)

    # ===== Model =====
    model = MiniGPT(
        vocab_size=vocab_size,
        seq_len=args.seq_len,
        n_layer=args.n_layer,
        n_head=args.n_head,
        n_embd=args.n_embd,
        n_ff=args.n_ff,
        attn_pdrop=args.attn_pdrop,
        resid_pdrop=args.resid_pdrop,
    )
    # tie weights
    model.lm_head.weight = model.tok_emb.weight
    model.to(device)
    print("Model params:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

    # ===== Optimizer & scheduler =====
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    # Simple LR decay (optional)
    total_steps = args.epochs * len(dataloader)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=args.lr, total_steps=max(1, total_steps)) if args.onecycle else None

    # ===== Training =====
    model.train()
    for epoch in range(args.epochs):
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{args.epochs}", leave=False)
        total_loss = 0.0
        for step, (x, y) in enumerate(pbar):
            x = x.to(device)
            y = y.to(device)
            logits = model(x)  # (B, T, V)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))
            optimizer.zero_grad()
            loss.backward()
            # gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
            optimizer.step()
            if scheduler:
                scheduler.step()

            total_loss += loss.item()
            if (step + 1) % args.log_interval == 0:
                avg = total_loss / args.log_interval
                pbar.set_postfix_str(f"loss={avg:.4f}")
                total_loss = 0.0

        # epoch end: save checkpoint
        out_dir = Path(args.output_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        ckpt_path = out_dir / f"mini_gpt_epoch{epoch+1}.pt"
        torch.save({"model_state_dict": model.state_dict(), "tokenizer_vocab": tokenizer.get_vocab()}, ckpt_path)
        print(f"Saved checkpoint: {ckpt_path}")

        # Optional sample generation
        model.eval()
        prompt = args.sample_prompt or "Once upon a time"
        tokens = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)["input_ids"].to(device)
        with torch.no_grad():
            gen = model.generate(tokens, max_new_tokens=args.sample_length, eos_token_id=None, temperature=args.sample_temperature)
        text_out = tokenizer.decode(gen[0].cpu().numpy(), skip_special_tokens=True)
        print("--- sample ---")
        print(text_out)
        print("--------------")
        model.train()

    print("Training complete.")


# ---------------------------
# CLI
# ---------------------------
# -------------------------------------
# CLI
# -------------------------------------
def parse_args():
  p = argparse.ArgumentParser()
  p.add_argument("--dataset", type=str, default="wikitext", help="HF dataset name (eg: 'wikitext')")
  p.add_argument("--dataset_config", type=str, default="wikitext-2-raw-v1", help="HF dataset config (e.g. 'wikitext-2-raw-v1')")
  p.add_argument("--text_file", type=str, default=None, help="Path to a single text file to train on (overrides dataset)")
  p.add_argument("--max_chars", type=int, default=2000000, help="Limit characters for quick runs (default 2M). Set to 0 for no limit.")
  p.add_argument("--seq_len", type=int, default=128)
  p.add_argument("--batch_size", type=int, default=32)
  p.add_argument("--epochs", type=int, default=3)
  p.add_argument("--lr", type=float, default=5e-4)
  p.add_argument("--weight_decay", type=float, default=0.01)
  p.add_argument("--grad_clip", type=float, default=1.0)
  p.add_argument("--n_layer", type=int, default=6)
  p.add_argument("--n_head", type=int, default=8)
  p.add_argument("--n_embd", type=int, default=512)
  p.add_argument("--n_ff", type=int, default=None)
  p.add_argument("--attn_pdrop", type=float, default=0.0)
  p.add_argument("--resid_pdrop", type=float, default=0.0)
  p.add_argument("--output_dir", type=str, default="./checkpoints")
  p.add_argument("--log_interval", type=int, default=20)
  p.add_argument("--onecycle", action="store_true", help="Use OneCycleLR scheduler")
  # sampling options
  p.add_argument("--sample_prompt", type=str, default="Once upon a time")
  p.add_argument("--sample_length", type=int, default=64)
  p.add_argument("--sample_temperature", type=float, default=1.0)

  # ==== FIX FOR COLAB / JUPYTER ====
  args, _ = p.parse_known_args()
  return args


if __name__ == "__main__":
  args = parse_args()
  # small tweak: if max_chars = 0 then no limit
  if args.max_chars == 0:
    args.max_chars = None

  train(args)


Using device: cuda


Token indices sequence length is longer than the specified maximum sequence length for this model (440294 > 1024). Running this sequence through the model will result in indexing errors


Total tokens in corpus: 440294
Model params: 44.700672 M


Epoch 1/3:   0%|          | 0/107 [00:00<?, ?it/s]

Saved checkpoint: checkpoints/mini_gpt_epoch1.pt
--- sample ---
Once upon a time Russian occurred argued Wheeler with 1938 . 
 cute Christmass of the main billion since often to being creat . During the first @-@ single than Lock electors from the Shaw . After the Palestiniant as Beth and animal Hotia , defeated his Bob Civil Maurice isero , Ram Historyudrie 's families and been
--------------


Epoch 2/3:   0%|          | 0/107 [00:00<?, ?it/s]

Saved checkpoint: checkpoints/mini_gpt_epoch2.pt
--- sample ---
Once upon a time through that motif church of thriller , as gradually life to pursue any result total feelings of cancer . fashionardi with cont by hilar the local Reach on 84 , although all cases that I non miles . 
 After the Spec conception , Sarnrinhric von congratorsor helped a evidence = = = = = =
--------------


Epoch 3/3:   0%|          | 0/107 [00:00<?, ?it/s]

Saved checkpoint: checkpoints/mini_gpt_epoch3.pt
--- sample ---
Once upon a time at the power . As hisize the 10 , fuel Wing forence , the precise @-@ satirical Bridges in a second season , and another and the season share , but she joined the playoffs in An Western home season blue from the season where beat more pairs . In least 1914 ) , he eliminated by her deals and
--------------
Training complete.
